In [1]:
import os
import json

from duckduckgo_search import DDGS

from langchain_groq import ChatGroq

from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
GROQ_API_KEY = "secret_key"

In [13]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

vector_db = Chroma(
    persist_directory="../chroma_db",
    embedding_function=embedding_model
)

retriever = vector_db.as_retriever(
    search_kwargs={"k": 5}
)

C:\Users\Divyansh\AppData\Local\Temp\ipykernel_8648\3344270270.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


In [5]:
ROUTER_PROMPT = """
Classify the user query.

Return ONLY one label:

RAG
SEARCH
GENERAL

Rules:

RAG:
- Brain tumors
- Symptoms
- Causes
- Treatments
- Medical information

SEARCH:
- Doctors
- Hospitals
- Specialists
- Nearby hospitals
- Recommendations

GENERAL:
- Greetings
- General conversation
- Anything else

User Query:
{query}
"""

In [6]:
def route_query(query):

    prompt = ROUTER_PROMPT.format(
        query=query
    )

    response = llm.invoke(prompt)

    return response.content.strip()

In [7]:
def rag_tool(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
Use the context below to answer.

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)

    return response.content

In [8]:
def search_tool(query):

    results = []

    with DDGS() as ddgs:

        search_results = ddgs.text(
            query,
            max_results=10
        )

        for item in search_results:

            results.append({

                "title":
                    item.get("title", ""),

                "body":
                    item.get("body", ""),

                "url":
                    item.get("href", "")

            })

    prompt = f"""
Answer the question using
the search results.

Question:
{query}

Results:
{json.dumps(results, indent=2)}
"""

    response = llm.invoke(prompt)

    return response.content

In [10]:
def general_tool(query):

    response = llm.invoke(query)

    return response.content

In [11]:
def medical_chatbot(query):

    route = route_query(query)

    print(
        f"\n[ROUTER]: {route}\n"
    )

    if "RAG" in route:

        return rag_tool(query)

    elif "SEARCH" in route:

        return search_tool(query)

    else:

        return general_tool(query)

In [15]:
print(
    medical_chatbot(
        "What are symptoms of glioma?"
    )
)


[ROUTER]: RAG

According to the context, symptoms of glioma (specifically brain stem glioma) include:

1. Weakness or sensation changes of the face
2. Swallowing difficulty
3. Hoarseness
4. Weakness, loss/changes in sensation, or poor coordination on one side of the body
5. Headaches
6. Nausea
7. Vomiting
8. Unsteadiness in walking

Additionally, glioblastomas (a type of glioma) can cause symptoms such as:

1. Nausea
2. Vomiting
3. Severe headaches (typically worse in the morning)
4. Weakness or sensory changes of the face, arm, or leg
5. Balance difficulties
6. Neurocognitive/memory issues
7. Seizures

It's worth noting that symptoms can vary depending on the location and size of the tumor, and may be unique to each individual.
